<a href="https://colab.research.google.com/github/XW-ABAP/docs-l10n/blob/master/site/zh-cn/tutorials/quickstart/advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2019 The TensorFlow Authors.

In [1]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 专家的 TensorFlow 2 快速入门

<table class="tfo-notebook-buttons" align="left">
  <td>     <a target="_blank" href="https://tensorflow.google.cn/tutorials/quickstart/advanced"><img src="https://tensorflow.google.cn/images/tf_logo_32px.png">View on TensorFlow.org</a>
</td>
  <td><a target="_blank" href="https://colab.research.google.com/github/tensorflow/docs-l10n/blob/master/site/zh-cn/tutorials/quickstart/advanced.ipynb"><img src="https://tensorflow.google.cn/images/colab_logo_32px.png">在 Google Colab 中运行</a></td>
  <td>     <a target="_blank" href="https://github.com/tensorflow/docs-l10n/blob/master/site/zh-cn/tutorials/quickstart/advanced.ipynb"><img src="https://tensorflow.google.cn/images/GitHub-Mark-32px.png">在 GitHub 上查看源代码</a>
</td>
  <td>     <a href="https://storage.googleapis.com/tensorflow_docs/docs-l10n/site/zh-cn/tutorials/quickstart/advanced.ipynb"><img src="https://tensorflow.google.cn/images/download_logo_32px.png">下载笔记本</a>   </td>
</table>

这是一个 [Google Colaboratory](https://colab.research.google.com/notebooks/welcome.ipynb) 笔记本。Python 程序可以直接在浏览器中运行，这是学习和使用 Tensorflow 的绝佳方式。要按照本教程操作，请点击此页面顶部的按钮，在 Google Colab 中运行笔记本。

1. 在 Colab 中，连接到 Python 运行时：在菜单栏的右上方，选择 *CONNECT*。
2. 运行所有笔记本代码单元：选择 *Runtime* &gt; *Run all*。

下载并安装 TensorFlow 2。将 TensorFlow 导入您的程序：

注：升级 `pip` 以安装 TensorFlow 2 软件包。请参阅[安装指南](https://tensorflow.google.cn/install)了解详细信息。

导入tensorflow 和model

将 TensorFlow 导入到您的程序：

In [2]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)

from tensorflow.keras.layers import Dense, Flatten, Conv2D
from tensorflow.keras import Model

TensorFlow version: 2.21.0


加载并准备 [MNIST 数据集](http://yann.lecun.com/exdb/mnist/)。

In [13]:
# 得到这个mnist模型
mnist = tf.keras.datasets.mnist
# 获取x y的数据
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

# Add a channels dimension
x_train = x_train[..., tf.newaxis].astype("float32")
x_test = x_test[..., tf.newaxis].astype("float32")

使用 `tf.data` 将数据集分批和重排：

In [14]:
# 把训练数据改成缓存区域10000大小 一批次有32个
# 例如总共有60000个数据 这个时候会60000/32
train_ds = tf.data.Dataset.from_tensor_slices(
    (x_train, y_train)).shuffle(10000).batch(32)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(32)

使用 Keras <a>模型子类化 API</a> 构建 <code>tf.keras</code> 模型：

在 Keras 的 Conv2D 中，如果没有显式设置参数，它默认使用：步长（Stride） = 1（卷积核每次移动 1 个像素）填充（Padding） = 'valid'（不进行边缘填充）核心计算公式计算卷积后输出尺寸的公式为：$$\text{Output Size} = \frac{\text{Input Size} - \text{Kernel Size} + 2 \times \text{Padding}}{\text{Stride}} + 1$$把我们的数值代入进去：$\text{Input Size} = 28$ （输入图片的宽高）$\text{Kernel Size} = 3$ （卷积核大小是 $3 \times 3$）$\text{Padding} = 0$ （不填充）$\text{Stride} = 1$ （每次走 1 步）$$\text{Output Size} = \frac{28 - 3 + 0}{1} + 1 = 25 + 1 = 26$$所以，输出的宽高就变成了 $26 \times 26$。形象的理解：为什么会变小？想象一下，你手里有一张 $28 \times 28$ 的网格纸，拿一个 $3 \times 3$ 的框去盖在上面：如果把框放在最左上角，它刚好覆盖第 1 到第 3 行/列。如果把框一直往右滑动，滑到最右边时，它只能停在第 26 到第 28 行/列（因为再往右就掉出纸外了）。因此，从左到右、从上到下总共只能放下 26 个不同的中心起始位置。这就是为什么尺寸从 28 变到了 26。如果你希望卷积后尺寸保持不变（依然是 $28 \times 28$），可以在代码中加上 padding='same'：

为了让你彻底看清 `Flatten` 到底干了什么，我们用一个**极度简化的小网格**来模拟一张数字图片。

假设经过 `Conv2D` 提取特征后，输出的不再是 32 个通道，而是为了方便观察，我们假设只有 **2 个通道**，且每个通道的大小只有 **$3 \times 3$** 像素。

---

### 第一步：想象卷积输出的“立体方块”

此时的数据形状是 `(3, 3, 2)`（即：高 3、宽 3、通道数 2）。
它在计算机内存里看起来像两张重叠的 $3 \times 3$ 表格（像两层夹心饼干）：

* **第 1 个通道（Channel 1，比如检测竖线的特征图）**：

$$\begin{bmatrix} 0 & 5 & 0 \\  0 & 5 & 0 \\  0 & 5 & 0 \end{bmatrix}$$



*(能看出中间有一竖排数字 5，代表这里检测到了竖直线)*
* **第 2 个通道（Channel 2，比如检测横线的特征图）**：

$$\begin{bmatrix} 0 & 0 & 0 \\  8 & 8 & 8 \\  0 & 0 & 0 \end{bmatrix}$$



*(能看出中间有一横排数字 8，代表这里检测到了横线)*

---

### 第二步：`Flatten()` 开始干活（按顺序“一字排开”）

`Flatten` 的工作规则很简单：**先读完第一层通道的所有数字（从左到右、从上到下），再接着读第二层通道的所有数字，把它们连成一条线。**

我们跟着它的视角一行行把数字抠出来：

1. **先扫第一层（Channel 1）**：
* 第一行：`0, 5, 0`
* 第二行：`0, 5, 0`
* 第三行：`0, 5, 0`
*(按顺序串起来：`0, 5, 0, 0, 5, 0, 0, 5, 0`)*


2. **接着扫第二层（Channel 2）**：
* 第一行：`0, 0, 0`
* 第二行：`8, 8, 8`
* 第三行：`0, 0, 0`
*(按顺序串起来：`0, 0, 0, 8, 8, 8, 0, 0, 0`)*



---

### 第三步：最终输出的一维向量

把上面两段长条拼接在一起，就成了 `Flatten()` 的最终输出结果：

$$\begin{aligned} [0, 5, 0,\ 0, 5, 0,\ 0, 5, 0,\ 0, 0, 0,\ 8, 8, 8,\ 0, 0, 0] \end{aligned}$$

* **形状变化**：从原本立体的 **`(3, 3, 2)`** 变成了一个长为 **`18`** 的一维数组（因为 $3 \times 3 \times 2 = 18$）。

---

### 回到你真实的模型代码中

在你原代码的那个例子里：

* 经过 `Conv2D(32, 3)` 后，数据形状是 **$(26 \times 26 \times 32)$**。
* 经过 `Flatten()` 之后，它会把这 32 层、每层 26×26 的所有格子全部按这个规则拆开、连成一条线。
* 最终变成一个长度为 **$26 \times 26 \times 32 = 21,632$** 的一维数字长龙，直接喂给后面的全连接层（`self.d1 = Dense(128)`）。

这两行代码是整个模型的“大脑与决策层”（全连接层，Fully Connected Layer）。

如果说前面的 `Conv2D` 负责“找局部笔画”，`Flatten` 负责“把笔画排成一队”，那么这两层 `Dense` 就负责**综合所有的线索，并最终猜出这张图片到底是个什么东西**。

我们可以把它们拆解开来看看：

---

### 1. 第一层：`self.d1 = Dense(128, activation='relu')` —— 综合思考与特征浓缩

* **`Dense`（全连接层）**：
在这一层里，每一个神经元都和上一层输出的所有数字（比如前面拉直出来的 21,632 个数字）相连。它就像是一个“超级委员会”，每个人都要听取所有特征的信息。
* **`128`（128 个神经元）**：
它把庞大、杂乱的一维特征进行**高度浓缩和重组**，提炼出 128 个更高级的抽象概念。
* 例如：有的神经元专门负责琢磨“有没有长横加长斜杠”；有的神经元专门负责琢磨“有没有封闭的圆圈”。


* **`activation='relu'`**：
再次使用 ReLU 激活函数，过滤掉负面的、无效的组合，只把有价值的高级特征传递给下一层。

---

### 2. 第二层：`self.d2 = Dense(10)` —— 最终的判决与输出

这是模型的**最后一层（输出层）**：

* **`Dense`**：同样是全连接层。
* **`10`（10 个神经元）**：
**为什么是 10？** 因为这是一个经典的 **10 分类任务**（例如识别手写数字 0 到 9，总共有 10 个可能的类别）。
* 这里的这 10 个神经元，每一个分别代表一个数字（第 0 个代表“0”，第 1 个代表“1”……第 9 个代表“9”）。


* **没有设置激活函数**：
它会直接输出 10 个原始分数（在深度学习中叫 **Logits**）。
* 比如输入一张“7”的图片后，经过层层计算，这 10 个神经元分别给出了自己的“嫌疑得分”：
* 数字 0 的神经元：`-1.2`
* 数字 7 的神经元：`8.5` （分数最高！）
* 数字 3 的神经元：`0.4`
* ……


* 最后，计算机只需要看谁的分数最高，就会宣判：“**我猜这张图片是数字 7！**”



---

### 通俗总结整个大结局

当一张写着“7”的图片喂进这个模型时：

1. **`conv1`** 找出了横线、竖线等笔画特征。
2. **`flatten`** 把特征排成一长条队伍。
3. **`d1`**（128个神经元）把这些笔画组合起来思考：“有横、有斜竖……这可能是一个几何图形”。
4. **`d2`**（10个神经元）进行最终投票：“在 0 到 9 这十个候选里，‘7’的票数最高，所以它就是 7！”

In [15]:
# 继承了父类Model
class MyModel(Model):
  def __init__(self):
    # 调用父类Model初始化数据
    super().__init__()
    # 32是卷积核的数量 也就是通道的数量 3代表卷积核的大小为3*3
    # 使用relu激活函数 假设一张28*28的像素的单通道黑白照片（28*28*1）
    # 用3*3的小矩阵带着9个格子去扫描 从左上的位置开始每次覆盖9个格子
    # 有32个这样的3*3小矩阵去扫描 分别负责找例如竖线 横线 左上角的拐角 等待
    # 实际上3*3去扫描的时候 有个卷积核公式
    self.conv1 = Conv2D(32, 3, activation='relu')
    self.flatten = Flatten()
    self.d1 = Dense(128, activation='relu')
    self.d2 = Dense(10)

  def call(self, x):
    x = self.conv1(x)
    x = self.flatten(x)
    x = self.d1(x)
    return self.d2(x)

# Create an instance of the model
model = MyModel()

选择用于训练的优化器和损失函数：

In [16]:
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

optimizer = tf.keras.optimizers.Adam()

选择指标来衡量模型的损失和准确率。这些指标在周期内累积值，然后打印总体结果。

In [17]:
train_loss = tf.keras.metrics.Mean(name='train_loss')
train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

test_loss = tf.keras.metrics.Mean(name='test_loss')
test_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='test_accuracy')

使用 `tf.GradientTape`训练模型：

In [18]:
@tf.function
def train_step(images, labels):
  with tf.GradientTape() as tape:
    # training=True is only needed if there are layers with different
    # behavior during training versus inference (e.g. Dropout).
    predictions = model(images, training=True)
    loss = loss_object(labels, predictions)
  gradients = tape.gradient(loss, model.trainable_variables)
  optimizer.apply_gradients(zip(gradients, model.trainable_variables))

  train_loss(loss)
  train_accuracy(labels, predictions)

测试模型：

In [19]:
@tf.function
def test_step(images, labels):
  # training=False is only needed if there are layers with different
  # behavior during training versus inference (e.g. Dropout).
  predictions = model(images, training=False)
  t_loss = loss_object(labels, predictions)

  test_loss(t_loss)
  test_accuracy(labels, predictions)

In [12]:
EPOCHS = 5

for epoch in range(EPOCHS):
  # Reset the metrics at the start of the next epoch
  train_loss.reset_state()
  train_accuracy.reset_state()
  test_loss.reset_state()
  test_accuracy.reset_state()

  for images, labels in train_ds:
    train_step(images, labels)

  for test_images, test_labels in test_ds:
    test_step(test_images, test_labels)

  print(
    f'Epoch {epoch + 1}, '
    f'Loss: {train_loss.result()}, '
    f'Accuracy: {train_accuracy.result() * 100}, '
    f'Test Loss: {test_loss.result()}, '
    f'Test Accuracy: {test_accuracy.result() * 100}'
  )

Epoch 1, Loss: 0.13613826036453247, Accuracy: 95.93000030517578, Test Loss: 0.06803997606039047, Test Accuracy: 97.87999725341797
Epoch 2, Loss: 0.04349428415298462, Accuracy: 98.68500518798828, Test Loss: 0.055534087121486664, Test Accuracy: 98.12999725341797
Epoch 3, Loss: 0.022559259086847305, Accuracy: 99.26499938964844, Test Loss: 0.06243927404284477, Test Accuracy: 98.1500015258789
Epoch 4, Loss: 0.01415837649255991, Accuracy: 99.5566635131836, Test Loss: 0.05952685326337814, Test Accuracy: 98.20999908447266
Epoch 5, Loss: 0.007966762408614159, Accuracy: 99.73833465576172, Test Loss: 0.05877234786748886, Test Accuracy: 98.37999725341797


现在，经过训练，照片分类器在此数据集上的准确率约为 98%。要了解详情，请阅读 [TensorFlow 教程](https://tensorflow.google.cn/tutorials)。